# HAI Data Cleaning — Training + Challenge

**Cleans and reshapes HAI data into participant-level wide format for both the training and challenge sets.**
- Input: `train_hai.tsv`, `challenge_hai.tsv` (long format — one row per participant × timepoint × strain)
- Output: `hai_cleaned.csv`, `challenge_hai_cleaned.csv` (wide format — one row per participant, log2 titer per strain-timepoint column)

## Design notes

**Log2 transform:** Applied to all titer values. Consistent with downstream modeling notebooks which operate in log2 space.

**Timepoint mapping (training only):** Day 30 is aliased to Day 28 (group means are nearly identical). Only days 0, 28, and 365 are retained.

**Challenge data:** Contains only Day 0 baseline measurements for 17 strains; no timepoint aliasing needed.

**Pivot:** Long rows are pivoted to wide format; column names follow `HAI_{strain}_d{day}`.

In [1]:
TIMEPOINTS_TO_KEEP = [0.0, 28.0, 365.0]
TIMEPOINT_ALIAS = {30.0: 28.0}

In [2]:
DATA_PATH = '../data'
CLEAN_DATA_PATH = '../cleaned_data'

In [3]:
import os

import numpy as np
import pandas as pd

## Training Data

In [4]:
df_train = pd.read_csv(DATA_PATH + '/train_hai.tsv', sep='\t')
print(f'Raw shape: {df_train.shape}')
df_train.head()

Raw shape: (128177, 6)


,hai_id,participant_id,timepoint,virus_strain,value,material
0,ID_001__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_001,0.0,H1N1 A/South Carolina/1/1918,20.,Unknown
1,ID_001__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_001,28.0,H1N1 A/South Carolina/1/1918,40.,Unknown
2,ID_002__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_002,0.0,H1N1 A/South Carolina/1/1918,5.,Unknown
3,ID_002__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_002,28.0,H1N1 A/South Carolina/1/1918,5.,Unknown
4,ID_003__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_003,0.0,H1N1 A/South Carolina/1/1918,80.,Unknown


### Drop unused columns, filter invalid strains, and coerce values

In [5]:
df_train = df_train.drop(columns=['hai_id', 'material'])
df_train = df_train[df_train['virus_strain'] != '-']
df_train['value'] = pd.to_numeric(df_train['value'], errors='coerce')
print(f'After dropping unused columns and filtering invalid strains: {df_train.shape}')

After dropping unused columns and filtering invalid strains: (127959, 4)


### Timepoint analysis

In [6]:
df_train['timepoint'].value_counts().sort_index()

timepoint
-7.0         18
 0.0      49017
 3.0        318
 7.0        136
 14.0       777
 28.0     48240
 30.0       468
 70.0       180
 75.0       318
 90.0      2202
 180.0       51
 365.0    23808
Name: count, dtype: int64

In [7]:
df_train.groupby('timepoint')['value'].mean()

timepoint
-7.0       47.777778
 0.0       73.792527
 3.0      160.424528
 7.0      208.851544
 14.0      28.403520
 28.0     142.261548
 30.0     141.481744
 70.0     149.000000
 75.0     317.751572
 90.0     137.563579
 180.0    144.705882
 365.0     83.095474
Name: value, dtype: float64

### Filter and alias timepoints

In [8]:
df_train['timepoint'] = df_train['timepoint'].replace(TIMEPOINT_ALIAS)
df_train = df_train[df_train['timepoint'].isin(TIMEPOINTS_TO_KEEP)]
print(f'After timepoint filtering: {df_train.shape}')
df_train.head()

After timepoint filtering: (121533, 4)


,participant_id,timepoint,virus_strain,value
0,2016_UGA.ID_001,0.0,H1N1 A/South Carolina/1/1918,20.0
1,2016_UGA.ID_001,28.0,H1N1 A/South Carolina/1/1918,40.0
2,2016_UGA.ID_002,0.0,H1N1 A/South Carolina/1/1918,5.0
3,2016_UGA.ID_002,28.0,H1N1 A/South Carolina/1/1918,5.0
4,2016_UGA.ID_003,0.0,H1N1 A/South Carolina/1/1918,80.0


### Pivot to wide format and apply log2 transform

In [9]:
df_train['hai_timepoint'] = (
    'HAI_' + df_train['virus_strain'].astype(str)
    + '_d' + df_train['timepoint'].astype(int).astype(str)
)

df_train_pivot = df_train.pivot_table(
    index='participant_id',
    columns='hai_timepoint',
    values='value'
)
df_train_pivot = df_train_pivot.reset_index()
df_train_pivot = df_train_pivot.rename_axis(None, axis=1)

for col in df_train_pivot.columns:
    if col != 'participant_id':
        df_train_pivot[col] = df_train_pivot[col].apply(lambda x: np.log2(x) if pd.notna(x) else x)

print(f'Pivoted shape: {df_train_pivot.shape}')
df_train_pivot

Pivoted shape: (3757, 196)


,participant_id,HAI_Anc B/Lee/1940_d0,HAI_Anc B/Lee/1940_d28,HAI_Anc B/Lee/1940_d365,HAI_Anc B/Maryland/1959_d0,HAI_Anc B/Maryland/1959_d28,HAI_Anc B/Singapore/1964_d0,HAI_Anc B/Singapore/1964_d28,HAI_H1N1 A/Beijing/262/1995_d0,HAI_H1N1 A/Beijing/262/1995_d28,...,HAI_Yam B/Sichuan/379/1999_d365,HAI_Yam B/Texas/6/2011_d0,HAI_Yam B/Texas/6/2011_d28,HAI_Yam B/Texas/6/2011_d365,HAI_Yam B/Wisconsin/1/2010_d0,HAI_Yam B/Wisconsin/1/2010_d28,HAI_Yam B/Wisconsin/1/2010_d365,HAI_Yam B/Yamagata/16/1988_d0,HAI_Yam B/Yamagata/16/1988_d28,HAI_Yam B/Yamagata/16/1988_d365
0,2016_UGA.ID_001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.321928,5.321928,...,NaN,7.321928,8.321928,NaN,8.321928,8.321928,NaN,7.321928,8.321928,NaN
1,2016_UGA.ID_002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.321928,5.321928,...,NaN,5.321928,6.321928,NaN,6.321928,6.321928,NaN,5.321928,5.321928,NaN
2,2016_UGA.ID_003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.321928,5.321928,...,NaN,8.321928,8.321928,NaN,8.321928,8.321928,NaN,8.321928,8.321928,NaN
3,2016_UGA.ID_004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.321928,5.321928,...,NaN,4.321928,6.321928,NaN,5.321928,6.321928,NaN,3.321928,5.321928,NaN
4,2016_UGA.ID_005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.321928,2.321928,...,6.321928,6.321928,6.321928,5.321928,6.321928,6.321928,5.321928,5.321928,5.321928,5.321928
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3752,SDY887.SUB134259,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3753,SDY887.SUB134260,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3754,SDY887.SUB197783,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3755,SDY887.SUB197784,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
os.makedirs(CLEAN_DATA_PATH, exist_ok=True)
df_train_pivot.to_csv(CLEAN_DATA_PATH + '/hai_cleaned.csv', index=False)
print(f'Saved to {CLEAN_DATA_PATH}/hai_cleaned.csv')

Saved to ../cleaned_data/hai_cleaned.csv


## Challenge Data

In [11]:
df_chal = pd.read_csv(DATA_PATH + '/challenge_hai.tsv', sep='\t')
print(f'Raw shape: {df_chal.shape}')
df_chal.head()

Raw shape: (680, 6)


,hai_id,participant_id,timepoint,virus_strain,value,material
0,ID_077__2024_UGA_High_Dose_Fluzone__0__HAI__H1...,2024_UGA.ID_077,0,H1N1 A/California/7/2009,20,Unknown
1,ID_077__2024_UGA_High_Dose_Fluzone__0__HAI__H1...,2024_UGA.ID_077,0,H1N1 A/Brisbane/2/2018,5,Unknown
2,ID_077__2024_UGA_High_Dose_Fluzone__0__HAI__H1...,2024_UGA.ID_077,0,H1N1 A/Guangdong-Maonan/SWL1536/2019,5,Unknown
3,ID_077__2024_UGA_High_Dose_Fluzone__0__HAI__H1...,2024_UGA.ID_077,0,H1N1 A/Victoria/2570/2019,5,Unknown
4,ID_077__2024_UGA_High_Dose_Fluzone__0__HAI__H1...,2024_UGA.ID_077,0,H1N1 A/Victoria/4897/2022,5,Unknown


### Drop unused columns, filter timepoints, and coerce values

In [12]:
df_chal = df_chal.drop(columns=['hai_id', 'material'])
df_chal = df_chal[df_chal['virus_strain'] != '-']
df_chal = df_chal[df_chal['timepoint'].isin(TIMEPOINTS_TO_KEEP)]
df_chal['value'] = pd.to_numeric(df_chal['value'], errors='coerce')
print(f'After filtering: {df_chal.shape}')

After filtering: (680, 4)


### Virus strain coverage

In [13]:
df_chal['virus_strain'].value_counts()

virus_strain
H1N1 A/California/7/2009                40
H1N1 A/Brisbane/2/2018                  40
H1N1 A/Guangdong-Maonan/SWL1536/2019    40
H1N1 A/Victoria/2570/2019               40
H1N1 A/Victoria/4897/2022               40
H3N2 A/Hong Kong/4801/2014              40
H3N2 A/Singapore/INFIMH-160019/2016     40
H3N2 A/Kansas/14/2017                   40
H3N2 A/Hong Kong/2671/2019              40
H3N2 A/South Australia/34/2019          40
H3N2 A/Tasmania/503/2020                40
H3N2 A/Darwin/9/2021                    40
H3N2 A/Massachusetts/18/2022            40
Vic B/Colorado/6/2017                   40
Vic B/Washington/2/2019                 40
Vic B/Austria/1359417/2021              40
Yam B/Phuket/3073/2013                  40
Name: count, dtype: int64

### Pivot to wide format and apply log2 transform

In [14]:
df_chal['hai_timepoint'] = (
    'HAI_' + df_chal['virus_strain'].astype(str)
    + '_d' + df_chal['timepoint'].astype(int).astype(str)
)

df_chal_pivot = df_chal.pivot_table(
    index='participant_id',
    columns='hai_timepoint',
    values='value'
)
df_chal_pivot = df_chal_pivot.reset_index()
df_chal_pivot = df_chal_pivot.rename_axis(None, axis=1)

for col in df_chal_pivot.columns:
    if col != 'participant_id':
        df_chal_pivot[col] = df_chal_pivot[col].apply(lambda x: np.log2(x) if pd.notna(x) else x)

print(f'Pivoted shape: {df_chal_pivot.shape}')
df_chal_pivot

Pivoted shape: (40, 18)


,participant_id,HAI_H1N1 A/Brisbane/2/2018_d0,HAI_H1N1 A/California/7/2009_d0,HAI_H1N1 A/Guangdong-Maonan/SWL1536/2019_d0,HAI_H1N1 A/Victoria/2570/2019_d0,HAI_H1N1 A/Victoria/4897/2022_d0,HAI_H3N2 A/Darwin/9/2021_d0,HAI_H3N2 A/Hong Kong/2671/2019_d0,HAI_H3N2 A/Hong Kong/4801/2014_d0,HAI_H3N2 A/Kansas/14/2017_d0,HAI_H3N2 A/Massachusetts/18/2022_d0,HAI_H3N2 A/Singapore/INFIMH-160019/2016_d0,HAI_H3N2 A/South Australia/34/2019_d0,HAI_H3N2 A/Tasmania/503/2020_d0,HAI_Vic B/Austria/1359417/2021_d0,HAI_Vic B/Colorado/6/2017_d0,HAI_Vic B/Washington/2/2019_d0,HAI_Yam B/Phuket/3073/2013_d0
0,2024_UGA.ID_077,2.321928,4.321928,2.321928,2.321928,2.321928,2.321928,4.321928,3.321928,3.321928,2.321928,3.321928,6.321928,3.321928,4.321928,2.321928,2.321928,4.321928
1,2024_UGA.ID_086,8.321928,9.321928,9.321928,6.321928,4.321928,2.321928,2.321928,2.321928,2.321928,2.321928,2.321928,4.321928,3.321928,5.321928,5.321928,4.321928,5.321928
2,2024_UGA.ID_128,6.321928,7.321928,6.321928,6.321928,5.321928,6.321928,7.321928,7.321928,7.321928,7.321928,7.321928,8.321928,7.321928,3.321928,3.321928,3.321928,5.321928
3,2024_UGA.ID_170,3.321928,3.321928,3.321928,5.321928,2.321928,3.321928,5.321928,3.321928,4.321928,7.321928,4.321928,5.321928,6.321928,3.321928,2.321928,2.321928,3.321928
4,2024_UGA.ID_179,5.321928,3.321928,2.321928,6.321928,2.321928,4.321928,3.321928,2.321928,6.321928,3.321928,2.321928,4.321928,3.321928,5.321928,4.321928,4.321928,7.321928
5,2024_UGA.ID_215,2.321928,3.321928,2.321928,2.321928,2.321928,5.321928,7.321928,6.321928,6.321928,4.321928,6.321928,8.321928,8.321928,4.321928,5.321928,4.321928,3.321928
6,2024_UGA.ID_219,6.321928,6.321928,6.321928,5.321928,2.321928,3.321928,4.321928,2.321928,3.321928,4.321928,2.321928,4.321928,4.321928,5.321928,7.321928,7.321928,6.321928
7,2024_UGA.ID_275,3.321928,2.321928,5.321928,3.321928,3.321928,2.321928,3.321928,2.321928,3.321928,2.321928,2.321928,3.321928,4.321928,4.321928,4.321928,4.321928,7.321928
8,2024_UGA.ID_295,6.321928,2.321928,5.321928,6.321928,4.321928,2.321928,2.321928,2.321928,4.321928,3.321928,2.321928,4.321928,2.321928,9.321928,5.321928,4.321928,4.321928
9,2024_UGA.ID_296,5.321928,4.321928,4.321928,3.321928,2.321928,2.321928,2.321928,2.321928,2.321928,2.321928,2.321928,4.321928,2.321928,4.321928,3.321928,4.321928,3.321928


In [15]:
os.makedirs(CLEAN_DATA_PATH, exist_ok=True)
df_chal_pivot.to_csv(CLEAN_DATA_PATH + '/challenge_hai_cleaned.csv', index=False)
print(f'Saved to {CLEAN_DATA_PATH}/challenge_hai_cleaned.csv')

Saved to ../cleaned_data/challenge_hai_cleaned.csv
